In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
num_qubits = 3
dev = qml.device('default.qubit', wires=num_qubits)

In [3]:
G1 = qml.PauliZ(0) @ qml.PauliZ(1)
G2 = qml.PauliZ(1) @ qml.PauliZ(2)
P = qml.PauliZ(0) @ qml.PauliZ(1) @ qml.PauliZ(2)
H = -1.0 * P - 1.0 * G1 - 1.0 * G2

In [4]:
def ansatz(params):
    qml.StronglyEntanglingLayers(weights=params, wires=range(num_qubits))

In [5]:
@qml.qnode(dev)
def cost_fn(params):
    ansatz(params)
    return qml.expval(H)

In [6]:
layers = 2
shape = qml.StronglyEntanglingLayers.shape(n_layers=layers, n_wires=num_qubits)
params = np.random.random(shape, requires_grad=True)

In [7]:
opt = qml.AdamOptimizer(stepsize=0.1)
max_iterations = 400

In [10]:
for i in range(max_iterations):
    params, energy = opt.step_and_cost(cost_fn, params)
    
    if (i + 1) % 40 == 0:
        print(f"params is {params} and cost is {energy}")
print(f"\nOptimization completed Final Energy: {cost_fn(params)} (Target: -3.0)")

params is [[[ 3.24794444e-01 -1.07136233e-11  3.80934317e-01]
  [ 9.16166222e-01  7.01322615e-03  1.28636311e+00]
  [ 8.10224329e-01 -2.17709903e-11  1.16435929e+00]]

 [[ 1.34542335e+00  1.29413153e-11  6.34627371e-01]
  [ 8.68062144e-01 -3.48298189e-12  2.87410247e-01]
  [ 7.54990556e-01  4.47697769e-11  9.56898750e-01]]] and cost is -2.9999592073860057
params is [[[ 3.24794451e-01 -2.15863206e-12  3.80934317e-01]
  [ 9.16166235e-01 -3.61235902e-04  1.28636311e+00]
  [ 8.10224347e-01  8.58490481e-13  1.16435929e+00]]

 [[ 1.34542335e+00 -9.80598699e-15  6.34627369e-01]
  [ 8.68062144e-01  1.01305531e-12  2.87410240e-01]
  [ 7.54990556e-01 -1.01509176e-13  9.56898747e-01]]] and cost is -2.999998502992919
params is [[[ 3.24794468e-01 -1.16304836e-13  3.80934317e-01]
  [ 9.16166269e-01  1.58271505e-04  1.28636311e+00]
  [ 8.10224351e-01  1.03886832e-13  1.16435929e+00]]

 [[ 1.34542335e+00  1.94586868e-13  6.34627366e-01]
  [ 8.68062144e-01 -3.46348500e-13  2.87410234e-01]
  [ 7.5499055

In [11]:
@qml.qnode(dev)
def get_final_state(params):
    ansatz(params)
    return qml.probs(wires=range(num_qubits))

In [12]:
probabilities = get_final_state(params)
target_prob = probabilities[0]
print(f"\nProbability of measuring the logical state |000>: {target_prob * 100}%")


Probability of measuring the logical state |000>: 99.99942099682056%
